# Per-regime calibration (PIT) of a trained NLE

For every star of held-out simulated galaxies, at the true theta: per-star log p and the PIT of the flow base coordinate, binned by the galaxy's true sigma_los and the assigned error. A calibrated density has PIT mean 0.5, std 0.2887 and 0.27% of |z| > 3 in every bin. `nll_gauss` is the NLL of a Gaussian with the TRUE intrinsic dispersion convolved with the error.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, os.path.expanduser('~/my_modules'))
os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
MODELS = Path(os.path.expanduser('~/scratch/projects/sbi_dsph/trained_models/nle/nle_8p_ZhaoPlumCOM_v4'))
OUT = Path(os.path.expanduser('~/scratch/projects/nle_dsph'))

In [ ]:
RUN = MODELS / '37vbi58p'   # the model to test
SHARDS = [31]
N_GAL = 20000
ERRORS = [0.3, 1.0, 3.0]

In [ ]:
from diagnostics import pit_by_regime
sys.argv = ['pit', '--run-dir', str(RUN), '--shards', *map(str, SHARDS), '--n-gal', str(N_GAL), '--errors', *map(str, ERRORS)]
pit_by_regime.main()

In [ ]:
df = pd.read_csv(RUN / 'pit_by_regime.csv')
df = df[df.err > 0]
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for e, g in df.groupby('err'):
    x = np.sqrt(g.sigma_lo.clip(lower=0.1) * g.sigma_hi)
    ax[0].plot(x, g.pit_mean, 'o-', label=f'err {e} km/s'); ax[1].plot(x, g.pit_std, 'o-'); ax[2].plot(x, g.nll - g.nll_gauss, 'o-')
ax[0].axhline(0.5, color='k', lw=0.8); ax[1].axhline(0.2887, color='k', lw=0.8); ax[2].axhline(0, color='k', lw=0.8)
for a, t in zip(ax, ['PIT mean (0.5)', 'PIT std (0.2887)', 'NLL - Gaussian(true sigma) [nats/star]']):
    a.set_xscale('log'); a.set_xlabel('sigma_true bin centre [km/s]'); a.set_title(t); a.grid(alpha=.3)
ax[0].legend(); plt.show()